# 第五章： 蒙特卡洛方法（Monte Carlo Methods）  

## 介绍（Introduction）

在本次实验中，掌握蒙特卡洛方法估计价值函数的方法

***
# 涉及核心概念回顾： 参考课堂上ppt


##### 导入实验所需要的库

In [1]:
import random
import numpy as np
import pprint
import random

# 设置随机种子
random.seed(123)
np.random.seed(123) 

%matplotlib inline

<div class="alert alert-block alert-warning">

**主题1： GridWorld环境创建**
</div>

<img src="./gridworld.jpg" style="zoom:100%" />


【问题描述】
上图4x4的grid world, 阴影部分为终止状态。该网格世界包含两个终止状态。除了终止状态，奖励均为为-1；一旦到达终止状态，则奖励为1。
动作空间为：$$\mathcal{A} = \{\text{up, right, down, left}\}$$


***步骤1-1***：基于上述环境描述，实现环境接口, 补全缺失代码（将``__TODO__``换成对应代码）（10分）：

In [2]:
class GridWorldEnv(object):
    """
    Simple 4x4 GridWorld Env from figure 4.1 in the RL textbook

    Actions are up (0), right (1), down (2), left (3).

    Reward is -1 for every timestep until the terminal states are reached
    at which a reward of 5 is given.

    Actions that take the agent off the grid leave the agent unchanged.
    """
    def __init__(self):
        self.action_space = (0, 1, 2, 3) # up, right, down, left
        self.observation_space = ((x, y) for x in range(4) for y in range(4))
        # Initialize the grid
        self.grid = np.zeros([4,4])
        self.grid[0][0] = 1
        self.grid[3][3] = 1
        self.max_steps = 50
        self.step_count = 0

    def step(self, action):
        assert action in self.action_space
        
        # *******************  测试1-1（1）. 基于动作，更新状态位置 *******************
        new_pos = None 
        if action == 0: # up
            new_pos = (self.pos[0] - 1, self.pos[1])
        elif action == 1: # right
            new_pos = (self.pos[0], self.pos[1] + 1)
        elif action == 2: # down
            new_pos = (self.pos[0] + 1, self.pos[1])
        elif action == 3: # left
            new_pos = (self.pos[0], self.pos[1] - 1)
        # ******************* END  *******************
        
        # Make sure the agent doesn't fall off of the grid
        if self._onGrid(new_pos):
            self.pos = new_pos

        # Check if the agent is in a terminal position
        # Negative reward for every timestep except for terminal step
        done = False
        reward = -1
        if self._inTerminalPos():
            done = True
            reward = 5

        if self.step_count == self.max_steps:
            done = True

        self.step_count += 1

        return self.pos, reward, done, {}

    def _onGrid(self, pos):
        return pos[0] in range(0,4) and pos[1] in range(0,4)

    def _inTerminalPos(self):
        return self.grid[self.pos[0]][self.pos[1]] == 1

    def reset(self):
        # Reset step count
        self.step_count = 0
        # Start the agent in the top right corner
        self.pos = (0,3)
        return self.pos

    # Optionally allow start position to be set
    def setStartPos(self, pos):
        if self._onGrid(pos):
            self.pos = pos
        else:
            print("ERROR: Start position not on grid")

    def render(self):
        # Convert every element in the grid from a number to a string
        new_grid = list(
            map(lambda r:
                list(map(lambda c: str(c), r)),
            self.grid)
        )

        # Put an X where the agent currrently is
        new_grid[self.pos[0]][self.pos[1]] = 'X'

        pprint.pprint(new_grid)

##### ***步骤1-2*** 基于上述环境，创建相应环境实例 (5分)

In [3]:
# ****************************测试1-2（1）************************************************
# ****************************STEP1: 创建环境实例 env************************************************
env = GridWorldEnv()

# ****************************STEP2: 调用reset方法，对智能体初始位置进行初始化************************************************
env.reset()

# ****************************STEP3: 调用render函数可视化************************************************
env.render()
# ****************************END************************************************

[['1.0', '0.0', '0.0', 'X'],
 ['0.0', '0.0', '0.0', '0.0'],
 ['0.0', '0.0', '0.0', '0.0'],
 ['0.0', '0.0', '0.0', '1.0']]


**问题**：本实验开始前为什么要固定随机种子,再比较不同方法？

**回答**：
固定随机种子可以控制随机初始化、动作采样和轨迹生成带来的波动，使不同方法在尽量相同的随机条件下比较，提升实验的可复现性与公平性。否则方法间差异可能被随机噪声掩盖，难以判断真实性能差别。

### ***步骤1-3: 策略的实现 （10分）***

In [4]:
class Policy(object):
    """
    Represents a policy

    p_type: type of policy
    - d: deterministic
    - nd: non-deterministic
    - eg: epsilon-greedy

    epsilon: optional epsilon parameter for eg policy.
    """
    def __init__(self, p_type, epsilon=0.1):
        valid_policy_types = ['d', 'nd', 'eg']
        assert(p_type in valid_policy_types)
        self.p_type = p_type
        self.policy = {}
        self.epsilon = epsilon
        self.initPolicyValues()

    def initPolicyValues(self):
        for x in range(4):
            for y in range(4):
                if self.p_type == 'nd':
                    self.policy[(x,y)] = [0.25]*4 
                else:
                    self.policy[(x,y)] = random.randint(0,3)

    # *******************  测试： 实现策略动作选择 *******************
    def getPolicyAction(self, s):
        # 基于self.p_type，返回状态s的动作
        if self.p_type == 'd':
            return self.policy[s]
        elif self.p_type == 'nd':
            return self._getNonDetPolicyAction(s)
        elif self.p_type == 'eg':
            return self._getEpGreedyPolicyAction(s)
    # ******************* END  *******************
    
    def _getNonDetPolicyAction(self, s):
        return np.random.choice(
            np.arange(0,4),
            p=self.policy[s])

    def _getEpGreedyPolicyAction(self, s):
        if random.uniform(0,1) < self.epsilon:
            # Explore
            return random.choice(list(range(4)))
        else:
            # Exploit
            return self.policy[s]
   # *******************  测试： 实现策略策略更新 *******************
    def updateDetPolicyState(self, s, a):
        self.policy[s] = a

##### ***步骤1-3: 辅助函数实现 （15分）***

In [5]:

def average(lst):
    if len(lst) == 0:
        return 0
    return sum(lst) / len(lst)

def isFirstVist(s_t, t, trajectory):
    for timestep in trajectory:
        s,a,r = trajectory[timestep]
        if s == s_t:
            return timestep == t


def genEpisode(env, policy, es=False):
    obs = env.reset()
    done = False
    trajectory = {}
    t = 0

    # Generate episode with exploring starts
    if es:
        # *******************  测试：实现exploring starts *******************
        # Choose a random start state and action
        start_state = random.choice([(x, y) for x in range(4) for y in range(4) if (x, y) not in [(0, 0), (3, 3)]])
        start_action = random.choice(list(env.action_space))
        env.setStartPos(start_state)
        obs = start_state
        obs, reward, done, info = env.step(start_action)
        trajectory[t] = [start_state, start_action, reward]
        t += 1
        # ******************* END  *******************

    while not done:
        s = obs
        action = policy.getPolicyAction(s)
        obs, reward, done, info = env.step(action)
        trajectory[t] = [s,action,reward]
        t += 1
    return trajectory

# Prints state for the grid world
def printGridStateValues(V):
    grid = np.zeros([4,4])

    for state, value in V.items():
        x = state[0]
        y = state[1]
        grid[x,y] = value

    print("Value Function--------------------------")
    pprint.pprint(grid)
    print('\n')

# Gets the action that has the highest Q value for this state
def getMaxActionForState(s, Q):
    max_value = float('-inf')
    max_action = None
    
    for (state, action), value in Q.items():
        # *******************  测试1-3：基于Q贪心s对应选择最佳动作 *******************
        if state == s and value > max_value:
            max_value = value
            max_action = action
    return max_action

# Prints the policy as a grid of arrows
def printPolicy(policy):
    grid = np.zeros([4,4])

    for state, action in policy.policy.items():
        x = state[0]
        y = state[1]
        grid[x,y] = action

    # Convert actions to arrows
    arrow_grid = []
    for row_index, row in enumerate(grid):
        arrow_grid_row = []
        for col_index, action in enumerate(row):
            arrow_char = ''
            if (row_index == 0 and col_index == 0) or (row_index == 3 and col_index == 3):
                arrow_grid_row.append(arrow_char)
            else:
                if action == 0:
                    arrow_char = '↑'
                elif action == 1:
                    arrow_char = '→'
                elif action == 2:
                    arrow_char = '↓'
                elif action == 3:
                    arrow_char = '←'
                arrow_grid_row.append(arrow_char)
        arrow_grid.append(arrow_grid_row)

    print("Policy--------------------------")
    pprint.pprint(arrow_grid)
    print('\n')

<div class="alert alert-block alert-warning">

**主题2：蒙特卡洛方法**
</div>

蒙特卡洛方法：基于轨迹数据，完成价值评估和策略改进。
- 蒙特卡洛预测问题：估计某个策略下，智能体在某个状态或状态-动作的价值
- 蒙特卡洛控制问题：基于预测结果，完成策略提升

***测试2-1***：不同蒙特卡洛方法实现

设置num_episodes （采样轨迹数目）, gamma（折扣因子）等超参

In [6]:
num_episodes = 10000
gamma = 0.9

***测试2-2***： first visit MC方法 (10分)

实现了首次访问的蒙特卡洛预测算法。它使用一个随机策略生成多个回合轨迹，并根据回合的轨迹来更新状态值函数 V。

In [7]:

def firstVisitMCPred(gamma, num_episodes):
    # *******************  测试2-2（1）：初始化策略 *******************
    policy = Policy('nd')
    # ******************* END  *******************
     
    V = {} 
    returns = {} 
    for x in range(4):
        for y in range(4):
            V[(x,y)] = 0 
            returns[(x,y)] = [] 

    # *******************  测试2-2（2）：MC预测算法实现 *******************
    for _ in range(num_episodes):
        trajectory = genEpisode(env, policy)
        G = 0
        for t in reversed(trajectory):
            s_t, a_t, r_t_plus_1 = trajectory[t]
            G = gamma * G + r_t_plus_1
            if isFirstVist(s_t, t, trajectory):
                returns[s_t].append(G)
                V[s_t] = average(returns[s_t])
    # ******************* END  ******************* 
    
    return V

firstVisitMCPred(gamma, num_episodes)

# output 
# {(0, 0): 0,
#  (0, 1): -1.9875206301002522,
#  (0, 2): -5.148958290007456,
#  (0, 3): -6.071778775334375,
#  (1, 0): -1.882353453605291,
#  (1, 1): -4.250947682492099,
#  (1, 2): -5.32460914050645,
#  (1, 3): -5.243271902523106,
#  (2, 0): -5.094397458764803,
#  (2, 1): -5.164329498376454,
#  (2, 2): -4.280941634881388,
#  (2, 3): -2.145775773306661,
#  (3, 0): -5.817140135335783,
#  (3, 1): -5.0664499779332255,
#  (3, 2): -2.091910739044393,
#  (3, 3): 0}


{(0, 0): 0,
 (0, 1): -1.9875206301002617,
 (0, 2): -5.148958290007469,
 (0, 3): -6.071778775334612,
 (1, 0): -1.88235345360529,
 (1, 1): -4.250947682492086,
 (1, 2): -5.3246091405064835,
 (1, 3): -5.243271902523138,
 (2, 0): -5.094397458764831,
 (2, 1): -5.164329498376463,
 (2, 2): -4.280941634881379,
 (2, 3): -2.1457757733066734,
 (3, 0): -5.817140135335799,
 (3, 1): -5.06644997793325,
 (3, 2): -2.0919107390443856,
 (3, 3): 0}

**问题**
- first-visit MC 中“首次访问”的判定对象是状态还是状态-动作对？在当前代码实现里是哪一种？


- 为什么要从轨迹末尾反向计算回报 G？如果正向计算会遇到什么困难？


**回答**
- 理论上两种都可以定义：first-visit state-value 用“状态首次访问”，first-visit action-value 用“状态-动作首次访问”。本 notebook 当前 `isFirstVist(s_t, t, trajectory)` 仅按 `s_t` 判断，因此实现的是“按状态首次访问”。
- 从末尾反向计算可用递推 $G_t = R_{t+1} + \gamma G_{t+1}$，每步 $O(1)$ 更新，简单且高效。若正向计算，每个时刻都要累加未来整段奖励，通常需要重复遍历后续序列，计算量更大且实现更繁琐。

***测试2-3***： first visit MC, 探索性开始（exploration start），简称为MCES （10分）。



In [8]:
def MCES(gamma, num_episodes):
    policy = Policy('d')
    Q = {} 
    returns = {} 
    for x in range(4):
        for y in range(4):
            for action in range(4):
                Q[((x,y), action)] = 0 
                returns[((x,y), action)] = [] 

    for _ in range(num_episodes):
        # *******************  测试2-3（1）：生成一个回合 *******************
        trajectory = genEpisode(env, policy, es=True)
        G = 0
        for t in reversed(trajectory):
            s_t, a_t, r_t_plus_1 = trajectory[t]
            G = gamma * G + r_t_plus_1
            if isFirstVist(s_t, t, trajectory):
                returns[(s_t, a_t)].append(G)
                Q[(s_t, a_t)] = average(returns[(s_t, a_t)])
                policy.updateDetPolicyState(s_t, getMaxActionForState(s_t, Q))
        # ******************* END  *******************

    printPolicy(policy)

MCES(gamma, num_episodes)
# output 
# Policy--------------------------
# [['', '←', '←', '→'],
#  ['↑', '←', '←', '↓'],
#  ['↑', '←', '→', '↓'],
#  ['↑', '→', '→', '']]

Policy--------------------------
[['', '←', '←', '←'],
 ['↑', '←', '↑', '↓'],
 ['↑', '←', '↓', '↓'],
 ['↑', '→', '→', '']]




**问题**
- MCES 中 exploration start 的核心作用是什么？它解决了哪类探索不足问题？

- 为什么在 MCES 中可以使用确定性策略作为目标策略，但仍能完成有效学习？

- 若某些状态-动作对在采样中出现极少，Q 值更新会受到什么影响？你会如何改进？

**回答**
- exploration start 的核心是让每个回合从随机状态-动作对启动，保证所有状态-动作对都有非零被访问概率，缓解“策略早期偏置导致部分动作永远不被尝试”的探索不足。
- MCES 依赖 exploring starts 保证覆盖性，因此即使目标策略是确定性的，数据中仍能持续出现多样状态-动作样本，从而支持策略迭代收敛到更优策略。
- 低频状态-动作对会导致样本方差大、估计不稳定、收敛很慢。可改进为：增加采样回合、使用 $\epsilon$-soft/$\epsilon$-greedy 持续探索、采用函数逼近进行泛化、或用加权更新/基线降低方差。

***测试2-4***： on policy first visit MC control (10分)

In [9]:
def OnPolicyFirstVistMCControl(gamma, num_episodes, epsilon=0.1):
    # *******************  测试2-4（1）：初始化策略 *******************
    policy = Policy('eg', epsilon=epsilon)
    # ******************* END  ******************* 
    
    Q = {} 
    returns = {} 
    for x in range(4):
        for y in range(4):
            for action in range(4):
                Q[((x,y), action)] = 0 
                returns[((x,y), action)] = [] 

    # 根据指定的回合数进行循环
    for _ in range(num_episodes):
        # *******************  测试2-4（2）：生成一个回合并获得轨迹 *******************
        trajectory = genEpisode(env, policy)
        # ******************* END  ******************* 
        
        G = 0
        for t in reversed(trajectory):
            s_t, a_t, r_t_plus_1 = trajectory[t]
            G = gamma * G + r_t_plus_1
            if isFirstVist(s_t, t, trajectory):
                # *******************  测试2-4（3）：MC控制算法实现 *******************
                returns[(s_t, a_t)].append(G)
                Q[(s_t, a_t)] = average(returns[(s_t, a_t)])
                policy.updateDetPolicyState(s_t, getMaxActionForState(s_t, Q))
                # ******************* END  ******************* 

    printPolicy(policy)

OnPolicyFirstVistMCControl(gamma, num_episodes)

Policy--------------------------
[['', '←', '←', '←'],
 ['↑', '←', '←', '↓'],
 ['↑', '↓', '→', '↓'],
 ['→', '→', '↑', '']]




**问题**

- 在 on-policy 控制中，行为策略与目标策略的关系是什么？

**回答**
在 on-policy 控制中，行为策略与目标策略是同一个策略（或同一参数化策略的同一步迭代版本）。也就是：用当前策略采样数据，再用这些数据改进当前策略本身。

***测试2-5***： off policy every visit MC control (15分)。

off-policy MC Control的增量实现, 使用加权重要性采样 （weighted importance sampling）

In [11]:

def OffPolicyEveryVistMCControl(gamma, num_episodes):
    
    # *******************  测试2-5（1）：初始化策略pi和b  *******************
    # 初始化目标策略为确定性策略，每个状态选择初始动作随机选择（初始化）
    pi = Policy('d') 
    # 行为策略为非确定性策略，每个状态的动作选择概率相等
    b = Policy('nd') 
    # ******************* END  ******************* 
 
    Q = {} 
    C = {} 
    for x in range(4):
        for y in range(4):
            for action in range(4):
                Q[((x,y), action)] = 0 
                C[((x,y), action)] = 0

    # 根据指定的回合数进行循环
    for _ in range(num_episodes):
        # 生成一个回合并获得轨迹
        # *******************  测试2-5（2）：生成一个回合并获得轨迹 *******************
        trajectory = genEpisode(env, b)
        # ******************* END  *******************
        
        G = 0
        W = 1
        for t in reversed(trajectory):
            s_t, a_t, r_t_plus_1 = trajectory[t]
            # *******************  测试2-5（3）：更新累积权重和动作值函数 *******************
            G = gamma * G + r_t_plus_1
            C[(s_t, a_t)] += W
            # ******************* END  *******************
            
            Q[(s_t, a_t)] += (W / C[(s_t, a_t)]) * (G - Q[(s_t, a_t)])
            pi.updateDetPolicyState(s_t, getMaxActionForState(s_t, Q))
   
            # 如果目标策略选择的动作不等于行为策略选择的动作，则跳出循环
            if a_t != pi.policy[s_t]:
                break
            
            W *= 1 / b.policy[s_t][a_t]
   
    printPolicy(pi)


OffPolicyEveryVistMCControl(gamma, num_episodes)

# output 
# Policy--------------------------
# [['', '↑', '→', '↓'],
#  ['↑', '↑', '←', '↓'],
#  ['↑', '←', '→', '↓'],
#  ['↑', '→', '→', '']]

Policy--------------------------
[['', '←', '←', '←'],
 ['↑', '←', '→', '↓'],
 ['↑', '→', '↓', '↓'],
 ['↑', '→', '→', '']]





**问题**

- off-policy 中为什么要区分行为策略 b 和目标策略 pi？


- importance sampling 权重 W 在更新中的直观含义是什么？


- 上述代码中，W *= 1 / b.policy[s_t][a_t]， 为什么每次乘上的分数，其分子设置为1

**回答**
- off-policy 的目标是“评估/改进一个策略 $\pi$”，但采样可以由另一个更易探索的策略 $b$ 产生。二者分离后，既能保持探索性，又能学习到目标策略价值。
- 权重 $W$ 反映“该条轨迹在目标策略下相对行为策略有多可能出现”，用于把由 $b$ 采到的数据重加权为对 $\pi$ 的无偏/一致估计。
- 这里目标策略 `pi` 是确定性策略：若轨迹动作与 `pi` 一致，则 $\pi(a|s)=1$；不一致时前面已经 `break`。因此比值 $\frac{\pi(a|s)}{b(a|s)}$ 的分子恒为 1，更新就写成乘 $\frac{1}{b(a|s)}$。

***测试2-6***： 总结（15分）。

- 请评价本实验中使用 epsilon-greedy 与 exploring starts 两种探索机制的优缺点。

- 如果将本实验从 4x4 GridWorld 扩展到更大状态空间，你认为 MC 方法会遇到哪些主要挑战？可行改进方向是什么？

- 结合本次实验，谈谈你对“基于采样的强化学习方法”理解上的最大收获与仍存在的疑问。

**回答**
- `epsilon-greedy` 优点是实现简单、可在线持续探索，适合 on-policy 控制；缺点是随机探索较盲目，后期仍会做次优动作。`exploring starts` 优点是理论上保证状态-动作覆盖；缺点是很多实际任务无法任意指定起始状态与起始动作，工程可行性较弱。
- 扩展到大状态空间后，主要挑战是采样效率低、回报方差大、很多状态-动作几乎访问不到、表格法存储和泛化能力不足。可行改进：函数逼近（线性/神经网络）、特征表示学习、分层或经验回放、结合 TD（如 MC-TD 折中）以提升数据利用率。
- 最大收获是：即使没有环境模型，只靠完整回合采样也能做价值估计与策略改进；“采样分布”决定学习效果。仍有疑问是：在高方差 off-policy 场景下，如何在保证收敛理论的同时更稳定地降低重要性采样方差。